## 1. Data Overview

In [ ]:
# 1.1 Load Data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../1_data/raw/Airline_review.csv")
print("Data loaded successfully!")

In [ ]:
# 1.2 Shape & Data Types
print("=== Shape ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

In [ ]:
# 1.3 Data Structure
df.head()

In [ ]:
# 1.4 Missing Values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df.sort_values('missing_%', ascending=False)

> ### **Note — Skytrax Review Data Characteristics**
> 
> The table below summarizes the input fields collected from the Skytrax airline review submission form and their characteristics. **This explains the reason that there are substantial missing values in certain columns.**
> 
> For reference, see the original review form: [Skytrax Review Form](https://www.airlinequality.com/write-a-review/?type=airline)

| Column | Scale | Notes |
|---|---|---|
| Airline Name | - | 497 unique airlines |
| Review_Title | - | Supplementary for analysis |
| Review Date | - | Scraping date, not flight date |
| Verified | True/False | Whether e-ticket or boarding pass was submitted |
| Review | 150~3500 chars | **Primary column for text analysis** |
| Aircraft | - | Optional field; high missingness |
| Type Of Traveller | - | Business / Family / Couple / Solo |
| Seat Type | - | First / Business / Premium Economy / Economy |
| Route | - | Free-text input |
| Date Flown | - | Actual flight date |
| Seat Comfort | 1~5 | Required field |
| Cabin Staff Service | 1~5 | Required field |
| Food & Beverages | 1~5 + N/A | N/A = service not available |
| Ground Service | 1~5 | Required field |
| Inflight Entertainment | 1~5 + N/A | N/A = service not available |
| Wifi & Connectivity | 1~5 + N/A | N/A = service not available |
| Value For Money | 1~5 | Required field |
| Overall_Rating | 1~10 | Different scale from sub-ratings |
| Recommended | Yes/No | **Target variable** |

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# 2.1 Target Variable (Recommended)
print("=== Recommended ===")
print(df['Recommended'].value_counts())
print(df['Recommended'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# 2.2 Known Categorical Columns
print("=== Verified ===")
print(df['Verified'].value_counts())

In [ ]:
print("=== Seat Type ===")
print(df['Seat Type'].value_counts(dropna=False))

In [ ]:
print("=== Type Of Traveller ===")
print(df['Type Of Traveller'].value_counts(dropna=False))

In [ ]:
# 2.3 Columns Requiring Investigation
# 2.3.1 Overall Rating
df["Overall_Rating"].value_counts(dropna=False)

> ### **Why Drop `Overall_Rating`?**
> 
> - **Missing 10**: Skytrax reviews use a 1 to 10 scale for `Overall_Rating`. The absence of 10-point ratings is unexplained and cannot be verified from the raw data alone.
> 
> - **Ambiguous 'n' values**: 842 entries contain 'n' which likely represents None or N/A, but cannot be reliably imputed or removed without introducing bias.
> 
> - **Data Leakage Risk**: `Overall_Rating` is logically correlated with the target variable `Recommended` (e.g., low ratings likely map to "no", high ratings to "yes"), which would inflate model performance and obscure the true predictive power of text-based sentiment features.

In [ ]:
# 2.3.2 Review Date (by Year)
df['Review Date'] = pd.to_datetime(df['Review Date'], format='mixed', errors='coerce')
df['Review Date'].dt.year.value_counts().sort_index()

In [ ]:
# 2.3.3 Date Flown (by Year)
df['Date Flown'] = pd.to_datetime(df['Date Flown'], format='%b-%y', errors='coerce')
df['Date Flown'].dt.year.value_counts().sort_index()

In [ ]:
# 2.3.4 Airline Name
print(f"Total unique airlines: {df['Airline Name'].nunique()}")
df['Airline Name'].value_counts().head(10)

In [ ]:
# 2.3.5 Aircraft
print(f"Total unique aircraft: {df['Aircraft'].nunique()}")
df['Aircraft'].value_counts().head(10)

In [ ]:
# 2.3.6 Route
print(f"Total unique routes: {df['Route'].nunique()}")
df['Route'].value_counts().head(10)

In [ ]:
# 2.4 Numerical Rating Columns
rating_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
               'Ground Service', 'Inflight Entertainment',
               'Wifi & Connectivity', 'Value For Money']

df[rating_cols].describe()

In [ ]:
# 2.5 Wifi & Connectivity - detailed check
df["Wifi & Connectivity"].value_counts(dropna=False)